In [22]:
# ==========================================
#  패키지 로드 및 구글 드라이브 마운트
# ==========================================
!pip install pyreadstat statsmodels

from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
target_cols = [
    'age',
    'sex',
    'kstrata',
    'psu',
    'wt_ntr',
    'genertn',
    'HEI',
    'incm',
    'edu',
    'BD1_11',
    'pa_aerobic',
]

df, meta = pyreadstat.read_sas7bdat(
    f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols
)
if 'HEI' in df.columns:
  df['HE_khei'] = df['HEI']

# 1인 가구 재코딩
df['genertn_rc'] = df['genertn'].apply(
    lambda x: 1.0 if x == 1.0 else (0.0 if (2.0 <= x <= 7.0) else np.nan)
)

# 조건 설정
cond_adult = df['age'] >= 20
cond_wt = df['wt_ntr'].notnull()
cond_exp = df['genertn_rc'].notnull()
cond_out = df['HE_khei'].notnull()

# 도메인 표시변수 subpop (1: 최종분석대상, 0: 제외)
df['subpop'] = np.where(
    cond_adult & cond_wt & cond_exp & cond_out, 1.0, 0.0
)

print(f'✅ 데이터 로드 및 subpop 정의 완료! 전체: {len(df):,}명')
# ==========================================
#  패키지 로드 및 구글 드라이브 마운트
# ==========================================
!pip install pyreadstat statsmodels

from google.colab import drive
import numpy as np
import pandas as pd
import pyreadstat
import statsmodels.api as sm

drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data'
target_cols = [
    'age',
    'sex',
    'kstrata',
    'psu',
    'wt_ntr',
    'genertn',
    'HEI',
    'incm',
    'edu',
    'BD1_11',
    'pa_aerobic',
]

df, meta = pyreadstat.read_sas7bdat(
    f'{DATA_DIR}/hn24_all.sas7bdat', usecols=target_cols
)
if 'HEI' in df.columns:
  df['HE_khei'] = df['HEI']

# 1인 가구 재코딩
df['genertn_rc'] = df['genertn'].apply(
    lambda x: 1.0 if x == 1.0 else (0.0 if (2.0 <= x <= 7.0) else np.nan)
)

# ---------------------------------------------------------
# 💡 [추가된 피드백 반영] BD1_11(음주 빈도)의 코드 9(무응답)를 결측치로 처리
# ---------------------------------------------------------
if 'BD1_11' in df.columns:
  df['BD1_11'] = df['BD1_11'].replace(9.0, np.nan)

# 조건 설정
cond_adult = df['age'] >= 20
cond_wt = df['wt_ntr'].notnull()
cond_exp = df['genertn_rc'].notnull()
cond_out = df['HE_khei'].notnull()

# 도메인 표시변수 subpop (1: 최종분석대상, 0: 제외)
df['subpop'] = np.where(
    cond_adult & cond_wt & cond_exp & cond_out, 1.0, 0.0
)

print(f'✅ 데이터 로드 및 subpop 정의 완료! 전체: {len(df):,}명')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 데이터 로드 및 subpop 정의 완료! 전체: 6,997명
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 데이터 로드 및 subpop 정의 완료! 전체: 6,997명


In [23]:
# ==========================================
#  B-1 단계별 제외 인원 정확한 수치 출력
# ==========================================
n_step1 = len(df)  # 6,997
n_step2 = cond_adult.sum()  # 6,002 (제외: 995)
n_step3 = (cond_adult & cond_wt).sum()  # 5,828 (제외: 174)
n_step4 = (cond_adult & cond_wt & cond_exp).sum()  # 5,819 (제외: 9)
n_final = int(df['subpop'].sum())  # 5,816 (제외: 3)

print('=' * 50)
print(' [B-1 검토의견 반영 정확한 흐름도 수치] ')
print('=' * 50)
print(f'1단계 초기 표본: {n_step1:,}명')
print(f'  - 제외 (20세 미만): {n_step1 - n_step2:,}명')
print(f'2단계 20세 이상 성인: {n_step2:,}명')
print(f'  - 제외 (영양조사 가중치 결측): {n_step2 - n_step3:,}명')
print(f'3단계 가중치 유효 대상: {n_step3:,}명')
print(f'  - 제외 (1인 가구 여부 무응답/결측): {n_step3 - n_step4:,}명 (9명)')
print(f'4단계 노출 유효 대상: {n_step4:,}명')
print(f'  - 제외 (식생활평가지수 KHEI 결측): {n_step4 - n_final:,}명 (3명)')
print(f'✅ 최종 분석 대상자 (subpop == 1): {n_final:,}명')
print('=' * 50)

 [B-1 검토의견 반영 정확한 흐름도 수치] 
1단계 초기 표본: 6,997명
  - 제외 (20세 미만): 995명
2단계 20세 이상 성인: 6,002명
  - 제외 (영양조사 가중치 결측): 174명
3단계 가중치 유효 대상: 5,828명
  - 제외 (1인 가구 여부 무응답/결측): 9명 (9명)
4단계 노출 유효 대상: 5,819명
  - 제외 (식생활평가지수 KHEI 결측): 3명 (3명)
✅ 최종 분석 대상자 (subpop == 1): 5,816명


In [24]:
# =========================================
# 복합표본설계 기반 올바른 SE 계산 및 범주별 비가중 n / 유효 % 출력
# =========================================

def calc_survey_mean_se(df_full, var_name):
    """교수님 피드백 반영: 원자료 구조 보존 도메인 분석 및 올바른 설계 기반 SE 산출"""
    def _get_se(domain_mask, d):
        # (1) 도메인 밖은 행을 지우지 않고 가중치를 0으로 처리
        w = d['wt_ntr'].where(domain_mask & d[var_name].notna(), 0)
        y = d[var_name].fillna(0)

        sum_w = np.sum(w)
        if sum_w == 0:
            return 0.0, 0.0
        mean_val = np.sum(w * y) / sum_w

        # (2) 잔차 계산 및 층화/집락 분산 추정
        d_res = d.copy()
        d_res['res'] = w * (y - mean_val)
        psu_sum = d_res.groupby(['kstrata', 'psu'])['res'].sum().reset_index()

        # var(ddof=1)은 이미 n-1로 나눈 값이므로, 층별 집락 수(n)인 strata_counts를 곱함
        strata_var = psu_sum.groupby('kstrata')['res'].var(ddof=1).fillna(0)
        strata_counts = psu_sum.groupby('kstrata')['psu'].count()

        var_tot = np.sum(strata_var * strata_counts)
        se_val = np.sqrt(var_tot) / sum_w
        return mean_val, se_val

    # 전체, 1인 가구, 다인 가구 도메인 마스크 정의 (행 삭제 없음)
    sub_mask = (df_full['subpop'] == 1.0)
    g1_mask = sub_mask & (df_full['genertn_rc'] == 1.0)
    g0_mask = sub_mask & (df_full['genertn_rc'] == 0.0)

    m_tot, se_tot = _get_se(sub_mask, df_full)
    m_g1, se_g1 = _get_se(g1_mask, df_full)
    m_g0, se_g0 = _get_se(g0_mask, df_full)

    return (m_tot, se_tot), (m_g1, se_g1), (m_g0, se_g0)

# 최종 분석군
df_sub = df[df['subpop'] == 1.0].copy()

print('=' * 65)
print(' [1. 교수님 수식 보정 반영 연속형 변수 올바른 Mean ± SE] ')
print('=' * 65)
for v in ['age', 'HE_khei']:
    (mt, st), (m1, s1), (m0, s0) = calc_survey_mean_se(df, v)
    print(f'📌 [{v}]')
    print(f'  - 전체: {mt:.2f} ± {st:.2f}')
    print(f'  - 1인 가구: {m1:.2f} ± {s1:.2f}')
    print(f'  - 다인 가구: {m0:.2f} ± {s0:.2f}')

print('\n' + '=' * 65)
print(' [2. 범주형 공변량 비가중 n 및 유효 백분율(Valid %) 출력] ')
print('=' * 65)

cat_vars = ['sex', 'incm', 'edu', 'BD1_11', 'pa_aerobic']

for cv in cat_vars:
    print(f'\n🔹 변수: [{cv}] (전체 결측: {df_sub[cv].isnull().sum()}명)')

    for g_name, g_df in [
        ('전체', df_sub),
        ('1인 가구', df_sub[df_sub['genertn_rc'] == 1.0]),
        ('다인 가구', df_sub[df_sub['genertn_rc'] == 0.0])
    ]:
        valid_df = g_df.dropna(subset=[cv])
        counts = valid_df[cv].value_counts().sort_index()

        # 유효 응답 기준 가중 백분율
        w_sum = valid_df.groupby(cv)['wt_ntr'].sum()
        w_pct = (w_sum / valid_df['wt_ntr'].sum()) * 100

        # 유효 응답 기준 비가중 백분율
        unw_pct = (counts / len(valid_df)) * 100

        res_df = pd.DataFrame({
            '비가중 n': counts,
            '비가중 Valid %': unw_pct.round(2),
            '가중 Valid %': w_pct.round(2)
        })
        print(f'  [{g_name}] (유효 n = {len(valid_df)}명)')
        print(res_df.to_string())
        print('-' * 40)

 [1. 교수님 수식 보정 반영 연속형 변수 올바른 Mean ± SE] 
📌 [age]
  - 전체: 50.11 ± 0.48
  - 1인 가구: 52.60 ± 1.59
  - 다인 가구: 49.60 ± 0.39
📌 [HE_khei]
  - 전체: 58.78 ± 0.28
  - 1인 가구: 56.87 ± 0.77
  - 다인 가구: 59.17 ± 0.27

 [2. 범주형 공변량 비가중 n 및 유효 백분율(Valid %) 출력] 

🔹 변수: [sex] (전체 결측: 0명)
  [전체] (유효 n = 5816명)
     비가중 n  비가중 Valid %  가중 Valid %
sex                                
1.0   2476        42.57       49.76
2.0   3340        57.43       50.24
----------------------------------------
  [1인 가구] (유효 n = 1051명)
     비가중 n  비가중 Valid %  가중 Valid %
sex                                
1.0    395        37.58       47.88
2.0    656        62.42       52.12
----------------------------------------
  [다인 가구] (유효 n = 4765명)
     비가중 n  비가중 Valid %  가중 Valid %
sex                                
1.0   2081        43.67       50.14
2.0   2684        56.33       49.86
----------------------------------------

🔹 변수: [incm] (전체 결측: 21명)
  [전체] (유효 n = 5795명)
      비가중 n  비가중 Valid %  가중 Valid %
incm                